# H3 Police Response Coverage

This notebook computes a complete H3 response-coverage grid for Western Australia. The pipeline code lives in `src/wa_critical_infrastructure_map/response_coverage.py` (covered by `tests/test_response_coverage.py`); this notebook runs it step by step and shows what each step produces.

Spatial joins, attribute lookups, and aggregations run as SQL in [DuckDB](https://duckdb.org/) with the `spatial` extension. Every intermediate result is a table in `data/processed/Police_Response_Coverage.duckdb`, so a run can be explored afterwards with plain SQL.

The workflow:

1. Load and validate roads, stations, and LGA boundaries, projecting to GDA94 / MGA zone 50 (EPSG:28350) with `ST_Transform`. Road speeds come from a `road_speeds` lookup table joined on `NETWORK_TYPE`. The first run converts each raw file to GeoParquet with `ST_Read` + `COPY ... TO`, in `data/processed/cache/`, keyed on the file contents. Later runs read the columnar copy (about 1.8 s instead of 11.5 s, and 33 MB instead of 194 MB for the roads).
2. Dense-polyfill every H3 cell that intersects the WA LGA boundary union at **resolution 7** (cell centre inside the boundary). The coastline is split into tiles of at most 500 vertices first, which makes the polyfill ~100× faster with identical cells. The grid is cached in `data/processed/cache/`, keyed on the boundary file contents, so later runs skip the polyfill.
3. Build a weighted, undirected road graph (a scipy sparse matrix) using segment length and configurable travel speeds.
4. Snap each cell centroid to the nearest **station-reachable** road segment within 5 km with an `ST_DWithin` spatial join; if none, no snap → unreachable (orphaned Main Roads components are ignored). Exact distance ties go to the lowest `segment_id`.
5. Calculate shortest-road response times from every police station to each cell's assigned road segment with scipy's compiled Dijkstra, aggregating per cell in SQL.
6. Classify each cell by its fastest response time: green/yellow/orange/red for reachable times, **`unreachable`** when no station can reach the cell (including no station-reachable road within the snap cap). Red means reachable but >15 min only.

**Perth land-strict gate membership:** cells with no station-reachable road within 5 km are outside the strict land gate (same threshold as the snap cap)—see `docs/response-coverage-improvements.md`.
7. Export the complete coverage as FlatGeobuf (written by DuckDB's `COPY ... TO`, opens directly in QGIS) and PMTiles (tippecanoe `-Z0 -z12`, no tiny-polygon drop).

> **Important:** The source road data does not contain a travel-speed field. The speed profile in `DEFAULT_ROAD_SPEED_KMH` is configurable and should be reviewed before operational use.

## Environment setup

Run the following command from the repository root before opening this notebook:

```bash
uv sync
uv run python -m jupyter notebook
```

`uv sync` installs the project package and its dependencies. DuckDB downloads its `spatial` extension on the first run, so that run needs network access. The `tippecanoe` executable must be installed separately and should be available on the shell path.

In [ ]:
from pathlib import Path

from wa_critical_infrastructure_map import response_coverage as rc

REPOSITORY_ROOT = Path.cwd()
if not (REPOSITORY_ROOT / "data" / "raw").exists():
    REPOSITORY_ROOT = REPOSITORY_ROOT.parent

config = rc.CoverageConfig.for_repository(REPOSITORY_ROOT)
con = rc.connect(config)
config

In [ ]:
counts = rc.load_inputs(con, config)
for label, count in counts.items():
    print(f"{label.replace('_', ' ').capitalize()}: {count}")
print(f"Excluded road rows: {counts['road_rows'] - counts['valid_road_rows']}")

con.sql("""
    SELECT 'roads' AS layer, ST_Extent_Agg(projected_geom) AS projected_extent FROM roads
    UNION ALL
    SELECT 'stations', ST_Extent_Agg(projected_geom) FROM stations
""").show()

In [ ]:
# The graph and H3 cell assignment are intentionally run separately so that
# failures are easy to diagnose before the potentially larger response run.
graph = rc.build_road_graph(con)
station_sources = rc.create_station_sources(con)
station_reachable_nodes = rc.nodes_reachable_from_stations(graph, station_sources)
cell_count, grid_from_cache = rc.generate_full_grid(con, config)
assigned_count = rc.assign_h3_cells(con, station_reachable_nodes)

print(f"Graph nodes: {con.execute('SELECT count(*) FROM road_nodes').fetchone()[0]}")
print(f"Road segments: {con.execute('SELECT count(*) FROM segments').fetchone()[0]}")
print(f"Full-grid H3 cells (dense polyfill{', from cache' if grid_from_cache else ''}): {cell_count}")
print(f"Station-reachable graph nodes: {len(station_reachable_nodes)}")
print(f"Cells with station-reachable road snap (≤{config.max_snap_metres:.0f} m): {assigned_count}")
print(f"Cells with no station-reachable snap (unreachable via snap cap): {cell_count - assigned_count}")
print(f"Station-to-road sources: {len(station_sources)}")

In [ ]:
rc.compute_cell_station_minutes(con, graph)
rc.build_coverage(con, config)

con.sql("""
    SELECT response_class, count(*) AS cells
    FROM coverage
    GROUP BY response_class
    ORDER BY list_position(['green', 'yellow', 'orange', 'red', 'unreachable'], response_class)
""").show()

In [ ]:
paths = rc.export_coverage(con, config)
for kind, path in paths.items():
    print(f"Wrote {kind}: {path} ({path.stat().st_size:,} bytes)")

## Validation checklist

Before uploading the layer, verify:

- The coverage is a dense H3 res-7 polyfill of the WA boundary (no lattice holes in Perth CBD).
- Every cell has `h3_resolution`, a response class, and a finite response-time field (`unreachable` when `reachable_stations=0` / no station-reachable road within 5 km; red only for reachable >15 min).
- Tippecanoe used `-Z0 -z12` with `--no-tiny-polygon-reduction` and `--drop-rate=0`.
- The metadata records the speed assumptions, dense-fill method, and limitations.
- The PMTiles file can be opened with the map client.
- The output is reviewed with an accurate speed profile before operational use.

The generated files are written to `data/processed` and are excluded from source control by the existing data ignore rules. S3 upload remains a separate, explicit step -- not part of this notebook run.

## Explore the working database

Every intermediate step is a table in `data/processed/Police_Response_Coverage.duckdb` (`roads`, `segments`, `station_sources`, `h3_cells`, `cell_segments`, `cell_station_minutes`, `coverage`, …). Query it here through `con`, or close the connection (`con.close()`) and open the file with the DuckDB CLI, for example:

```sql
LOAD spatial;
SELECT response_class, count(*) FROM coverage GROUP BY ALL;
```